# LangGraph + Repository Intelligence Orchestrator (M20-aligned)

This notebook uses LangGraph as the **task-level sequential orchestrator**.
The application graph owns dynamic task selection, task dispatch, result validation,
retry routing, budget gates, report assembly, and final collaboration validation.

The repository continues to own the existing M19/M20 task contracts, executor,
role policy, usage measurement, evidence validation, status machine, and trace sink.
No parallel execution or checkpoint/recovery is introduced in this milestone.


## 1. Project path and imports

In [ ]:
from __future__ import annotations

import json
import os
import sys
from dataclasses import replace
from pathlib import Path
from typing import Any, TypedDict

from IPython.display import display
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

# Prefer an explicit environment override; otherwise use the current directory.
# Do not hard-code a developer-specific Windows path into the notebook.
PROJECT_ROOT = Path(
    os.environ.get("REPO_INTELLIGENCE_ROOT", str(Path.cwd()))
).expanduser().resolve()
SRC_ROOT = PROJECT_ROOT / "src"

if not (SRC_ROOT / "repo_intelligence").is_dir():
    raise FileNotFoundError(
        "Could not find src/repo_intelligence under PROJECT_ROOT="
        f"{PROJECT_ROOT}. Set REPO_INTELLIGENCE_ROOT to the repository root."
    )

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from repo_intelligence.llm.config import load_env
from repo_intelligence.llm.openai_client import OpenAILLMClient
from repo_intelligence.planning import (
    InvestigationPlan,
    build_planning_request,
    load_architecture_context,
    plan_from_response,
    validate_plan,
)
from repo_intelligence.orchestration import (
    Orchestrator,
    RoleAssignment,
    assign_roles,
)
from repo_intelligence.multiagent import TaskPayload
from repo_intelligence.agent import JsonFileTraceSink
from repo_intelligence.repository_query import (
    M17_ORCHESTRATION_CONFIG,
    OFFLINE_COMMIT,
    QueryRouter,
    build_architect,
    build_engine,
    build_flow_investigator,
    build_memory_context,
    build_memory_manager,
    build_visualizer,
    default_artifacts_root,
    live_bindings,
    read_snapshot_metadata,
    resolve_read_target,
    verify_subject_snapshot,
)

load_env()
print(f"Project root: {PROJECT_ROOT}")
from repo_intelligence.orchestration.langgraph_runner import (
    build_langgraph_orchestration_graph,
)


## 2. Resolve and verify the pinned subject repository

In [ ]:
repository_root, readable_target, resolve_error = resolve_read_target()

if resolve_error:
    raise RuntimeError(f"Could not resolve subject repository: {resolve_error}")

verified_commit, commit_source, verification_error = verify_subject_snapshot(
    repository_root
)
if verification_error:
    raise RuntimeError(
        "Subject snapshot verification failed; refusing to analyze it: "
        f"{verification_error}"
    )

snapshot_metadata = read_snapshot_metadata(repository_root)
repository_label = (
    snapshot_metadata.get("name")
    or repository_root.name
    or str(repository_root)
)
current_commit = verified_commit or OFFLINE_COMMIT

print(f"Repository root: {repository_root}")
print(f"Repository scope: {repository_label}")
print(f"Commit scope: {current_commit}")
print(f"Commit source: {commit_source}")
print(f"Readable entry target: {readable_target}")

## 3. Load Architecture Context and initialize shared runtime

In [ ]:
# The planner consumes the already-generated Architecture Context artifact.
architecture_context, architecture_context_record = load_architecture_context(
    output_root=default_artifacts_root()
)

# Keep the repository's existing MemoryManager / MemoryContext / ContextEngine
# and tool-binding construction. The notebook does not invent parallel memory.
memory_manager = build_memory_manager()
memory_context = build_memory_context(
    repository=repository_label,
    current_commit=current_commit,
)
context_engine = build_engine(repository_label)
trace_sink = JsonFileTraceSink()

tool_bindings = live_bindings(
    repository_root,
    output_root=default_artifacts_root(),
)

print("Architecture Context:", architecture_context_record)
print("Shared runtime initialized.")

## 4. Build the existing specialists, QueryRouter, and M20 Orchestrator

In [ ]:
# Separate LLM clients preserve the repository's specialist responsibilities.
planner_llm = OpenAILLMClient()
architect_llm = OpenAILLMClient()
flow_llm = OpenAILLMClient()
visualizer_llm = OpenAILLMClient()

architect = build_architect(
    llm=architect_llm,
    trace_sink=trace_sink,
    memory_manager=memory_manager,
    memory_context=memory_context,
    context_engine=context_engine,
    bindings=tool_bindings,
)

flow_investigator = build_flow_investigator(
    llm=flow_llm,
    trace_sink=trace_sink,
    memory_manager=memory_manager,
    memory_context=memory_context,
    context_engine=context_engine,
    bindings=tool_bindings,
)

visualizer = build_visualizer(
    llm=visualizer_llm,
    trace_sink=trace_sink,
    memory_manager=memory_manager,
    memory_context=memory_context,
    context_engine=context_engine,
)

# Actual Investigator dispatch uses this router.
router = QueryRouter(
    architecture=architect,
    flow=flow_investigator,
)

# M20: activate the real per-attempt task budget. The executor passes this
# limit to run_result(..., max_tool_calls=...), where the agent's pre-tool
# guard enforces it. Change the environment variable to tune the limit.
MAX_TASK_TOOL_CALLS = int(
    os.environ.get("REPO_INTELLIGENCE_MAX_TASK_TOOL_CALLS", "12")
)
if MAX_TASK_TOOL_CALLS < 0:
    raise ValueError("REPO_INTELLIGENCE_MAX_TASK_TOOL_CALLS must be >= 0")

# Preserve the repository's whole-run ceiling and add the M20 task-level cap.
M20_ORCHESTRATION_CONFIG = replace(
    M17_ORCHESTRATION_CONFIG,
    max_task_tool_calls=MAX_TASK_TOOL_CALLS,
)

orchestrator = Orchestrator(
    investigator=router,
    analyst=visualizer,
    trace_sink=trace_sink,
    config=M20_ORCHESTRATION_CONFIG,
)

print("Architecture investigator:", type(architect).__name__)
print("Flow investigator:", type(flow_investigator).__name__)
print("Analyst:", type(visualizer).__name__)
print("Run-level tool budget:", M20_ORCHESTRATION_CONFIG.max_total_tool_calls)
print("Per-attempt task tool budget:", M20_ORCHESTRATION_CONFIG.max_task_tool_calls)
print("QueryRouter and M20 Orchestrator initialized.")

## 5. Define LangGraph state

In [ ]:
class RepoIntelligenceState(TypedDict, total=False):
    objective: str
    plan: InvestigationPlan
    role_assignment: list[RoleAssignment]
    role_assignments: list[RoleAssignment]
    task: TaskPayload
    run: Any
    orchestration_run: Any
    execution_status: str
    execution_error: str


## 6. Planner node

In [ ]:
def planner_node(state: RepoIntelligenceState) -> dict:
    objective = (state.get("objective") or "").strip()
    if not objective:
        raise ValueError("state['objective'] must be a non-empty string")

    request = build_planning_request(
        objective,
        architecture_context=architecture_context,
    )
    response = planner_llm.complete(request)
    plan = plan_from_response(response)

    # Keep deterministic, application-owned plan validation.
    validate_plan(plan)

    print(f"Plan created: {len(plan.steps)} step(s)")
    for step in plan.steps:
        print(f"  {step.id}: {step.goal}")

    return {"plan": plan}

## 7. Role assignment node

In [ ]:
def role_assigner_node(state: RepoIntelligenceState) -> dict:
    plan = state["plan"]
    assignments = assign_roles(plan.steps)

    step_ids = [step.id for step in plan.steps]
    assignment_ids = [item.step_id for item in assignments]

    if len(step_ids) != len(set(step_ids)):
        raise ValueError("The plan contains duplicate step IDs.")
    if len(assignment_ids) != len(set(assignment_ids)):
        raise ValueError("Role assignment contains duplicate step IDs.")
    if set(assignment_ids) != set(step_ids):
        raise ValueError(
            "Role assignments must exactly cover plan step IDs. "
            f"Plan={step_ids}; assignments={assignment_ids}"
        )

    print("Preassigned roles:")
    for item in assignments:
        print(f"  {item.step_id}: {item.role} — {item.reason}")

    return {"role_assignment": assignments, "role_assignments": assignments}

## 8. Task-level routing happens during real execution

The `QueryRouter` is already injected as the investigator participant in the
configured executor. The graph does not run a separate preview as a substitute
for dispatch: every investigator task goes through the real executor boundary,
which calls the router for that task's prompt. The analyst uses the existing
visualizer participant.


In [ ]:
# No preview-only router node: actual routing is performed by the executor
# during each task attempt, so the chosen specialist is the one that runs.


## 9. Build the task payload and invoke the LangGraph-owned task lifecycle


In [ ]:
def task_payload_node(state: RepoIntelligenceState) -> dict:
    task = TaskPayload(
        question=state["objective"],
        repository_scope=repository_label,
        commit_scope=current_commit,
    )
    return {"task": task}


## 10. Compile the LangGraph workflow

Planning and deterministic role assignment remain explicit nodes. The compiled
application subgraph owns the task loop; there is no call to
`Orchestrator.execute(...)` in the graph path.


In [ ]:
builder = StateGraph(RepoIntelligenceState)

builder.add_node("planner", planner_node)
builder.add_node("assign_role", role_assigner_node)
builder.add_node("build_task", task_payload_node)
builder.add_node("task_orchestration", build_langgraph_orchestration_graph(orchestrator))

def summarize_run_node(state: RepoIntelligenceState) -> dict:
    run = state["run"]
    print(f"Orchestration status: {run.status}")
    print(f"Termination reason: {run.termination_reason}")
    print(f"Completed steps: {run.completed_steps}")
    print(f"Failed steps: {run.failed_steps}")
    print(f"Blocked steps: {run.blocked_steps}")
    print(f"Measured tool calls: {run.tool_calls_total}")
    return {
        "orchestration_run": run,
        "execution_status": run.status,
        "execution_error": run.termination_reason or "",
    }

builder.add_node("summarize_run", summarize_run_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "assign_role")
builder.add_edge("assign_role", "build_task")
builder.add_edge("build_task", "task_orchestration")
builder.add_edge("task_orchestration", "summarize_run")
builder.add_edge("summarize_run", END)

graph = builder.compile()
display(graph.get_graph().draw_mermaid())


## 11. Invoke the workflow and inspect this run

In [ ]:
objective = "How do agents and MCP servers interact?"
result = graph.invoke({"objective": objective})

print("\n==================== FINAL RUN SUMMARY ====================")
print("Status:", result.get("execution_status"))
print("Termination:", result.get("execution_error"))

run = result.get("run") or result.get("orchestration_run")
if run is None:
    raise RuntimeError("LangGraph completed without returning an orchestration run.")

print("Run ID:", run.run_id)
print("Validation status:", run.validation_status)
print("Final:", run.final)
print("Completed steps:", run.completed_steps)
print("Failed steps:", run.failed_steps)
print("Blocked steps:", run.blocked_steps)
print("Run tool calls:", run.tool_calls_total)

print("\nTask records:")
for task_record in run.tasks:
    usage = task_record.usage
    usage_total = task_record.usage_total
    print(
        f"  {task_record.task_id}: status={task_record.status}, "
        f"role={task_record.role}, attempts={task_record.attempts}, "
        f"last_attempt_tool_calls={getattr(usage, 'tool_calls', None)}, "
        f"cumulative_tool_calls={getattr(usage_total, 'tool_calls', None)}, "
        f"trace_id={task_record.agent_run_id}"
    )

if run.finding is not None:
    print("\nFinding:")
    display(run.finding)

if run.report is not None:
    print("\nMerged investigation report:")
    display(run.report)

print("\nRole assignments:")
for assignment in result.get("role_assignment", []):
    print(f"  {assignment.step_id}: {assignment.role} — {assignment.reason}")


## 12. Inspect only traces produced by this orchestration run

In [ ]:
# The Orchestrator records agent_run_id values for attempts belonging to this
# run. Filter by those IDs; do not dump every historical trace in runtime/.
agent_run_ids = {
    item.get("run_id")
    for item in (run.agent_runs or [])
    if item.get("run_id")
}
trace_root = PROJECT_ROOT / "runtime" / "agent_runs"

if not trace_root.exists():
    print(f"Trace directory not found: {trace_root}")
else:
    trace_files = sorted(trace_root.glob("*/trace.json"))
    matching = []

    for trace_path in trace_files:
        try:
            with trace_path.open(encoding="utf-8") as handle:
                trace = json.load(handle)
        except (OSError, json.JSONDecodeError) as exc:
            print(f"Could not read {trace_path}: {exc}")
            continue

        if trace.get("run_id") in agent_run_ids:
            matching.append((trace_path, trace))

    if not matching:
        print("No trace files matched the agent run IDs recorded for this run.")
        print("Recorded agent run IDs:", sorted(agent_run_ids))
    for trace_path, trace in matching:
        summary = trace.get("summary", {})
        print(f"\nTrace file: {trace_path}")
        print("Trace ID:", trace.get("run_id"))
        print("Started:", trace.get("started_at"))
        print("Completed:", trace.get("completed_at"))
        print("Tool calls executed:", summary.get("tool_calls_executed"))
        print("Tool calls blocked:", summary.get("tool_calls_blocked"))
        for event in trace.get("orchestration_events", []):
            print(
                " ",
                event.get("timestamp"),
                event.get("operation"),
                "step=", event.get("step_id"),
                "role=", event.get("agent_role"),
                "status=", event.get("status"),
            )

## 13. How to interpret the result

- `COMPLETED` means orchestration and deterministic collaboration validation accepted the result.
- `INCOMPLETE` can occur even when every step completed: the report or finding can explicitly declare that evidence is insufficient, or no analyst ran.
- The per-attempt budget is active because this notebook constructs `M20_ORCHESTRATION_CONFIG` with `max_task_tool_calls`; the executor forwards it to the participant's pre-tool guard.
- `tool_calls_total` counts measured executed calls, not blocked or merely requested calls.
- The graph loops over the actual plan and selects one eligible task at a time. It has no fixed number of investigator nodes and no parallel dispatch.
